In [ ]:
!pip install fastexcel deltalake

In [ ]:
def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_running_in_colab():
    %cd /content/drive/MyDrive/01_itch_pipeline/src

/content/drive/MyDrive/01_itch_pipeline/src


In [ ]:
import sys
import os
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import pandas as pd
import deltalake
from deltalake import write_deltalake
import polars as pl
import numpy as np
from google.colab import drive
from pathlib import Path
from urllib.request import urlretrieve
from urllib.parse import urljoin
import gzip
import shutil
from collections import namedtuple, Counter, defaultdict
from struct import unpack
from datetime import timedelta
from time import time
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns

from src.config import BASE_DIR, DATA_DIR, RAW_DIR, ORDERBOOK_DIR, MESSAGE_TYPES_DIR, URL, EVENTS_CODES, ENCODINGS, FORMATS, TRADES_DIR
from src.downloader import download_nasdaq
from src.parser import message_types_parser, alpha_parser, alpha_parser, store_messages, format_time, binary_parser, get_bar_stats
from src.visualizer import share_of_value_plot, dist_plot, order_shape_ts_plot, raw_tick_plot, price_volume_plot
from src.orderbook import get_messages, get_trades, order_book_builder, get_dist, order_agg

In [ ]:
# 1. download nasdaq ITCH data and unzip it
unzipped_file_name = download_nasdaq(url=URL, target_dir=RAW_DIR)

# 2. parse the binary file
message_types, message_labels, message_fields, fstring = message_types_parser(message_type_dir=MESSAGE_TYPES_DIR, formats=FORMATS)
alpha_formats, alpha_length = alpha_parser(message_types)
binary_parser(unzipped_file_name, message_fields, fstring, alpha_formats, event_codes=EVENTS_CODES)

# 3. parse the messages and trades, and re-build the order book
messages = get_messages()
trades = get_trades(messages)
order_book_builder(messages )

# 4. plot the largest shares of value over the whole market
share_of_value_plot()

stock ='AAPL'
date = '10302019'
# 5. plot the distribution of buy/sell orders of AAPL
dist_plot()

# 6. plot the order book dynamics of AAPL with time envolving
order_shape_ts_plot()

# 7. plot the tick chart of AAPL
raw_tick_plot()

# 8. plot the price versus volumes, accumulative volumes and accumulative dollar value
trades = pl.scan_delta(TRADES_DIR/f'{stock}')
time_bars = get_bar_stats(trades)
volume_bars = get_bar_stats(trades, bar='volume')
dollar_bars = get_bar_stats(trades, bar='dollar')

price_volume_plot(time_bars,
             suptitle=f'Time Bars | {stock} | {pd.to_datetime(date, format='%m%d%Y').date()}')
price_volume_plot(volume_bars,
             suptitle=f'Volume Bars | {stock} | {pd.to_datetime(date, format='%m%d%Y').date()}',bar='c_vol')
price_volume_plot(dollar_bars,
             suptitle=f'Dollar Bars | {stock} | {pd.to_datetime(date, format='%m%d%Y').date()}',bar='d_val')
